# Sentinel ML: Training Isolation Forest
This notebook trains an Isolation Forest model on the first batch of credit card transactions to detect potential fraud.

In [ ]:
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_recall_fscore_support
import joblib
import os
from datetime import datetime
import numpy as np

In [ ]:
def split_credit_card_data():
    input_file = 'data/creditcard.csv'
    
    if not os.path.exists(input_file):
        print(f"Error: {input_file} not found.")
        return

    print(f"Reading {input_file}...")
    df = pd.read_csv(input_file)
    
    # Split into 6 batches
    chunk_size = int(np.ceil(len(df) / 6))
    batches = [df.iloc[i:i + chunk_size] for i in range(0, len(df), chunk_size)]
    
    print(f"Total rows: {len(df)}")
    
    for i, batch in enumerate(batches):
        batch_num = i + 1
        output_file = f'data/batch_{batch_num:02d}.csv'
        batch.to_csv(output_file, index=False)
        print(f"Saved {output_file} with {len(batch)} rows.")

if __name__ == "__main__":
    split_credit_card_data()

In [ ]:

# Load batch 1
print("Loading data/batch_01.csv...")
df = pd.read_csv('data/batch_01.csv')

In [ ]:
# Features and labels
X = df.drop('Class', axis=1)
y_true = df['Class']

In [ ]:
# Fit Isolation Forest
print("Training IsolationForest...")
model = IsolationForest(contamination=0.0017, random_state=42)
model.fit(X)

In [ ]:
# Predict
# IsolationForest returns -1 for outliers, 1 for inliers.
# Map to 1 for fraud, 0 for normal.
y_pred = model.predict(X)
y_pred = [1 if p == -1 else 0 for p in y_pred]

In [ ]:
# Evaluate
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary')

print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1: {f1:.4f}")

In [ ]:
# Save model
os.makedirs('models', exist_ok=True)
model_path = 'models/isolation_forest_v1.joblib'
joblib.dump(model, model_path)
print(f"Model saved to {model_path}")

In [ ]:
# Log to registry.csv
log_data = {
    'date': [datetime.now().strftime('%Y-%m-%d %H:%M:%S')],
    'f1': [f1],
    'precision': [precision],
    'recall': [recall],
    'model_path': [model_path]
}
log_df = pd.DataFrame(log_data)

registry_path = 'registry.csv'
if not os.path.exists(registry_path):
    log_df.to_csv(registry_path, index=False)
else:
    log_df.to_csv(registry_path, mode='a', header=False, index=False)
print(f"Results logged to {registry_path}")

## PSI Drift Detection
We use the Population Stability Index (PSI) to monitor shifts in the distribution of anomaly scores. A PSI > 0.2 indicates a significant shift, triggering a retrain flag.

In [ ]:
def calculate_psi(expected, actual, buckets=10):
    """
    Calculate the PSI (Population Stability Index) between two distributions.
    """
    # Define breakpoints based on deciles of the expected distribution
    breakpoints = np.percentile(expected, np.arange(0, 101, 100 / buckets))
    breakpoints[0] = -np.inf
    breakpoints[-1] = np.inf
    
    # Calculate percentages of samples in each bucket
    expected_percents = np.histogram(expected, bins=breakpoints)[0] / len(expected)
    actual_percents = np.histogram(actual, bins=breakpoints)[0] / len(actual)
    
    # Clip to avoid division by zero or log(0)
    expected_percents = np.clip(expected_percents, 1e-6, None)
    actual_percents = np.clip(actual_percents, 1e-6, None)
    
    # PSI Formula: sum((actual - expected) * ln(actual / expected))
    psi_value = np.sum((actual_percents - expected_percents) * np.log(actual_percents / expected_percents))
    
    return psi_value

In [ ]:
# Load baseline scores (Batch 01)
df_baseline = pd.read_csv('data/batch_01.csv')
X_baseline = df_baseline.drop('Class', axis=1)
baseline_scores = model.decision_function(X_baseline)

# Compare with incoming batches
print(f"{'Batch':<10} | {'PSI':<10} | {'Status':<10}")
print("-" * 35)

for i in range(2, 7):
    batch_file = f'data/batch_{i:02d}.csv'
    df_batch = pd.read_csv(batch_file)
    X_batch = df_batch.drop('Class', axis=1)
    batch_scores = model.decision_function(X_batch)
    
    psi = calculate_psi(baseline_scores, batch_scores)
    status = "DRIFT!" if psi > 0.2 else "Stable"
    
    print(f"{f'Batch {i:02d}':<10} | {psi:<10.4f} | {status:<10}")